# Notebook 08 · Patrones operativos públicos

Versión pública basada en el dataset candidato pseudonimizado. Mantiene el objetivo del notebook: leer patrones de selección histórica frente al mínimo de coste y preparar hipótesis de política.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

In [2]:
import numpy as np
import pandas as pd

V2_PATH = DATA_PUBLIC / "dataset_modelo_proveedor_v2_candidates.csv"
TARGET_DATES = [pd.Timestamp(date_text).date() for date_text in ['2028-05-02', '2028-05-03', '2028-05-06', '2028-05-07', '2028-05-08']]

df = pd.read_csv(V2_PATH, low_memory=False)
df["fecha_evento"] = pd.to_datetime(df["fecha_evento"], errors="coerce").dt.date
df["coste_min_dia_proveedor"] = pd.to_numeric(df["coste_min_dia_proveedor"], errors="coerce")
df["rank_coste_dia_producto"] = pd.to_numeric(df["rank_coste_dia_producto"], errors="coerce")
df["target_elegido"] = pd.to_numeric(df["target_elegido"], errors="coerce").fillna(0).astype(int)
df_window = df[df["fecha_evento"].isin(TARGET_DATES)].copy()

print({"rows_total": len(df), "rows_window": len(df_window), "events_window": df_window["event_id"].nunique()})
display(df_window.head(10))

{'rows_total': 155946, 'rows_window': 515, 'events_window': 61}


,event_id,fecha_evento,albaran_id,linea_id,producto_canonico,terminal_compra,proveedor_elegido_real,proveedor_candidato,coste_min_dia_proveedor,rank_coste_dia_producto,...,precio_unitario_evento,importe_total_evento,dia_semana,mes,fin_mes,blocked_by_rule_candidate,block_reason_candidate,target_elegido,v2_run_id,v2_ts_utc
112884,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.72,1.0,...,1.05697,10766.47,3,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z
112885,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_020,1069.88,2.0,...,1.05697,10766.47,3,5,0,1,RULE_002,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112886,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_050,1077.41,3.0,...,1.05697,10766.47,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112887,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_049,1127.30,4.0,...,1.05697,10766.47,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112888,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_019,1131.39,5.0,...,1.05697,10766.47,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112889,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.72,1.0,...,1.05697,4308.96,3,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z
112890,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_020,1069.88,2.0,...,1.05697,4308.96,3,5,0,1,RULE_002,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112891,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_050,1077.41,3.0,...,1.05697,4308.96,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112892,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_049,1127.30,4.0,...,1.05697,4308.96,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z
112893,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_019,1131.39,5.0,...,1.05697,4308.96,3,5,0,0,NaN,0,RUN_556F2D793E50,2030-03-06T12:21:19Z


In [3]:
event_min = (
    df_window.groupby("event_id", as_index=False)["coste_min_dia_proveedor"]
    .min()
    .rename(columns={"coste_min_dia_proveedor": "event_min_cost"})
)

selected_rows = (
    df_window[df_window["target_elegido"] == 1]
    .drop_duplicates(subset=["event_id"])
    .merge(event_min, on="event_id", how="left")
)
selected_rows["selected_in_min_cost"] = np.isclose(
    selected_rows["coste_min_dia_proveedor"],
    selected_rows["event_min_cost"],
    equal_nan=False,
)
selected_rows["delta_chosen_vs_min_eur_m3"] = (
    selected_rows["coste_min_dia_proveedor"] - selected_rows["event_min_cost"]
)

display(selected_rows.head(20))
print(
    {
        "selected_events": int(selected_rows["event_id"].nunique()),
        "selected_in_min_cost_rate": round(float(selected_rows["selected_in_min_cost"].mean()), 4),
    }
)

,event_id,fecha_evento,albaran_id,linea_id,producto_canonico,terminal_compra,proveedor_elegido_real,proveedor_candidato,coste_min_dia_proveedor,rank_coste_dia_producto,...,mes,fin_mes,blocked_by_rule_candidate,block_reason_candidate,target_elegido,v2_run_id,v2_ts_utc,event_min_cost,selected_in_min_cost,delta_chosen_vs_min_eur_m3
0,EVENT_77479261DC29,2028-05-02,ALBARAN_756CD69E7971,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.72,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,1060.72,True,0.00
1,EVENT_66253EE1433B,2028-05-02,ALBARAN_756CD69E7971,LINE_07771A799E32,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.72,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,1060.72,True,0.00
2,EVENT_184934B34A75,2028-05-02,ALBARAN_756CD69E7971,LINE_4B983CB16D58,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
3,EVENT_FC0DFFC1D21D,2028-05-02,ALBARAN_756CD69E7971,LINE_ECBE43367EA4,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
4,EVENT_D392705DE7CE,2028-05-02,ALBARAN_8CB12404B62D,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.72,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,1060.72,True,0.00
5,EVENT_CCAC70D4C697,2028-05-02,ALBARAN_8CB12404B62D,LINE_07771A799E32,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
6,EVENT_9E9CB7F491EE,2028-05-02,ALBARAN_8CB12404B62D,LINE_4B983CB16D58,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
7,EVENT_57751E700FF5,2028-05-02,ALBARAN_8CB12404B62D,LINE_ECBE43367EA4,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
8,EVENT_2E1953CC8664,2028-05-02,ALBARAN_A271A8FC1E21,LINE_0CBDAF1D1C30,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00
9,EVENT_3708C19AD95A,2028-05-02,ALBARAN_A271A8FC1E21,LINE_07771A799E32,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.51,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.51,True,0.00


{'selected_events': 61, 'selected_in_min_cost_rate': 0.9016}


In [4]:
pattern_by_day = (
    selected_rows.groupby(["fecha_evento", "producto_canonico"], as_index=False)
    .agg(
        events=("event_id", "nunique"),
        proveedores=("proveedor_elegido_real", lambda s: sorted(set(s.dropna()))),
        hit_rate_vs_min=("selected_in_min_cost", "mean"),
        median_delta_vs_min=("delta_chosen_vs_min_eur_m3", "median"),
    )
    .sort_values(["fecha_evento", "producto_canonico"])
)
display(pattern_by_day)

pattern_by_terminal = (
    selected_rows.groupby(["producto_canonico", "terminal_compra"], as_index=False)
    .agg(
        events=("event_id", "nunique"),
        unique_selected=("proveedor_elegido_real", "nunique"),
        median_delta_vs_min=("delta_chosen_vs_min_eur_m3", "median"),
    )
    .sort_values(["events", "median_delta_vs_min"], ascending=[False, False])
)
display(pattern_by_terminal.head(20))

,fecha_evento,producto_canonico,events,proveedores,hit_rate_vs_min,median_delta_vs_min
0,2028-05-02,PRODUCT_002,3,[SUPPLIER_009],1.000000,0.00
1,2028-05-02,PRODUCT_003,13,[SUPPLIER_009],1.000000,0.00
2,2028-05-03,PRODUCT_002,1,[SUPPLIER_009],0.000000,129.47
3,2028-05-03,PRODUCT_003,7,[SUPPLIER_009],1.000000,0.00
4,2028-05-06,PRODUCT_002,2,[SUPPLIER_009],0.000000,114.22
5,2028-05-06,PRODUCT_003,11,[SUPPLIER_009],1.000000,0.00
6,2028-05-07,PRODUCT_002,9,"[SUPPLIER_009, SUPPLIER_028]",0.777778,0.00
7,2028-05-07,PRODUCT_003,7,[SUPPLIER_009],1.000000,0.00
8,2028-05-08,PRODUCT_002,1,[SUPPLIER_009],0.000000,114.26
9,2028-05-08,PRODUCT_003,7,[SUPPLIER_009],1.000000,0.00


,producto_canonico,terminal_compra,events,unique_selected,median_delta_vs_min
1,PRODUCT_003,TERMINAL_001,45,1,0.0
0,PRODUCT_002,TERMINAL_001,16,2,0.0


In [5]:
focus_candidates = (
    selected_rows.groupby("albaran_id", as_index=False)
    .agg(events=("event_id", "nunique"), litros=("litros_evento", "sum"))
    .sort_values(["events", "litros"], ascending=[False, False])
)
ALBARAN_FOCUS = focus_candidates.iloc[0]["albaran_id"] if not focus_candidates.empty else None
focus_df = selected_rows[selected_rows["albaran_id"].eq(ALBARAN_FOCUS)].copy() if ALBARAN_FOCUS else pd.DataFrame()

print({"albaran_focus": ALBARAN_FOCUS, "focus_rows": len(focus_df)})
display(focus_df.head(20))

{'albaran_focus': 'ALBARAN_8DF23B0BE65B', 'focus_rows': 4}


,event_id,fecha_evento,albaran_id,linea_id,producto_canonico,terminal_compra,proveedor_elegido_real,proveedor_candidato,coste_min_dia_proveedor,rank_coste_dia_producto,...,mes,fin_mes,blocked_by_rule_candidate,block_reason_candidate,target_elegido,v2_run_id,v2_ts_utc,event_min_cost,selected_in_min_cost,delta_chosen_vs_min_eur_m3
49,EVENT_2019102AFE5A,2028-05-07,ALBARAN_8DF23B0BE65B,LINE_0CBDAF1D1C30,PRODUCT_002,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,1060.87,8.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,945.45,False,115.42
50,EVENT_B2655BA0EDFD,2028-05-07,ALBARAN_8DF23B0BE65B,LINE_07771A799E32,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.26,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.26,True,0.00
51,EVENT_99DC314B3126,2028-05-07,ALBARAN_8DF23B0BE65B,LINE_4B983CB16D58,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.26,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.26,True,0.00
52,EVENT_71BF44CC383E,2028-05-07,ALBARAN_8DF23B0BE65B,LINE_ECBE43367EA4,PRODUCT_003,TERMINAL_001,SUPPLIER_009,SUPPLIER_009,775.26,1.0,...,5,0,0,NaN,1,RUN_556F2D793E50,2030-03-06T12:21:19Z,775.26,True,0.00


In [6]:
rule_rows = []
for idx, (_, row) in enumerate(pattern_by_terminal.head(5).iterrows(), start=1):
    rule_rows.append(
        {
            "rule_candidate_id": f"RULE_{idx:03d}",
            "if_condition_data": f"producto_canonico == '{row['producto_canonico']}' and terminal_compra == '{row['terminal_compra']}'",
            "support_events": int(row["events"]),
            "median_delta_eur_m3": float(row["median_delta_vs_min"]) if pd.notna(row["median_delta_vs_min"]) else None,
            "reading": "Priorizar revisión manual si el patrón concentra eventos y sobrecoste frente al mínimo.",
        }
    )

df_rule_candidates = pd.DataFrame(rule_rows)
display(df_rule_candidates)

qa_summary = {
    "window_rows": int(len(df_window)),
    "window_events": int(df_window["event_id"].nunique()),
    "selected_rows": int(len(selected_rows)),
    "rule_candidates": int(len(df_rule_candidates)),
}
print("QA PASSED")
print(qa_summary)

,rule_candidate_id,if_condition_data,support_events,median_delta_eur_m3,reading
0,RULE_001,producto_canonico == 'PRODUCT_003' and termina...,45,0.0,Priorizar revisión manual si el patrón concent...
1,RULE_002,producto_canonico == 'PRODUCT_002' and termina...,16,0.0,Priorizar revisión manual si el patrón concent...


QA PASSED
{'window_rows': 515, 'window_events': 61, 'selected_rows': 61, 'rule_candidates': 2}
